## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. It also installs `scikit-posthocs`, which this analysis needs, into the same session. You can read every line of the code below before you run it.

# Listing photos and views: Dunn's comparisons

**Kruskal-Wallis Test with Multiple Comparisons** · a worked example from the StatNex teaching guide

**The story.** A resale app looked at 21 listings for the same kind of jacket, posted with 1, 3, 5 or 10 photos, and counted how many people viewed each listing in its first week.

**The question.** Do listings with more photos get more views, and how many photos are enough?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `scikit-posthocs`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "scikit-posthocs"

## The table

This cell saves the table `06_nonparametric/kruskal_wallis.csv` (6 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
1 photo,3 photos,5 photos,10 photos
88.0,118.0,135.0,148.0
102.0,131.0,150.0,170.0
95.0,109.0,128.0,139.0
120.0,140.0,162.0,165.0
76.0,,141.0,158.0
110.0,,155.0,
""")

## Libraries

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
import scikit_posthocs
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per level. Text and blank cells are skipped, so levels may differ in size.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
levels = ["1 photo", "3 photos", "5 photos", "10 photos"]
groups = [pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float) for name in levels]
sizes = np.array([len(g) for g in groups])

## Summary by level

In [ ]:
# All values are ranked together (ties share the average rank); each level's mean rank
# is what the test compares.
ranks = stats.rankdata(np.concatenate(groups))
mean_ranks = [part.mean() for part in np.split(ranks, np.cumsum(sizes)[:-1])]
summary = pd.DataFrame({"n": sizes, "mean rank": mean_ranks}, index=levels).T
print(summary)

## Kruskal-Wallis test

In [ ]:
# SciPy's H carries the tie correction; it is compared with chi-square on k - 1 df.
h, p = stats.kruskal(*groups)
df = len(groups) - 1
print(f"\nchi-square = {h:.6f}, df = {df}, P = {p:.6g}")

## Multiple comparison: Dunn

In [ ]:
# Every pair of levels, on the mean ranks of the whole sample, the ranking the
# Kruskal-Wallis test used. z is the difference of two mean ranks over its standard
# error, sqrt(N(N + 1)/12 x (1/n1 + 1/n2) x C), where C = 1 - sum(t^3 - t)/(N^3 - N)
# corrects for ties; it is positive when the second level sits higher.
# scikit-posthocs judges each z against the standard normal distribution (two-sided)
# and adjusts the P values for the number of pairs by Bonferroni's method: each is
# multiplied by the number of pairs, up to 1 (Dunn 1964).
adjusted = scikit_posthocs.posthoc_dunn(groups, p_adjust="bonferroni").to_numpy()
total = sizes.sum()
_, tie_sizes = np.unique(np.concatenate(groups), return_counts=True)
correction = 1 - (tie_sizes**3 - tie_sizes).sum() / (total**3 - total)
rows = []
for i, j in combinations(range(len(groups)), 2):
    se = np.sqrt(total * (total + 1) / 12 * (1 / sizes[i] + 1 / sizes[j]) * correction)
    rows.append([levels[i], levels[j], (mean_ranks[j] - mean_ranks[i]) / se, adjusted[i, j]])
pairs = pd.DataFrame(rows, columns=["level 1", "level 2", "z", "P"])
print()
print(pairs.to_string(index=False))

## What the result says

The Kruskal-Wallis test compares groups by ranks, the positions of the listings when sorted by views. Its chi-square statistic is 14.89 and the P value is 0.002, so the photo counts differ somewhere. The P value is the chance of gaps this large if photos made no difference.

Dunn's test then compares every pair of photo counts. It uses the same ranks as the Kruskal-Wallis test and turns the gap between two mean ranks into a statistic z. Six pairs are tested, so each P value is multiplied by the number of pairs: the Bonferroni adjustment. Two pairs are significant: 1 against 5 photos (z = 2.88, P = 0.024) and 1 against 10 photos (z = 3.51, P = 0.003). The others are not: 3 against 10 photos has P = 0.254, and 5 against 10 photos has P = 1.

Five or ten photos clearly beat one. A difference between three and one, or between ten and five, is not shown. That is not proof that they are equal: the listings are too few to settle it.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.